# Download das bibliotecas

In [1]:
!pip install pyswarm

  Preparing metadata (setup.py) ... done
  Created wheel for pyswarm: filename=pyswarm-0.6-py3-none-any.whl size=4463 sha256=66e4ad1cc348bd7da4c74421865d950a90b845c12f7862358ba1532de6bac89b
  Stored in directory: /root/.cache/pip/wheels/bb/4f/ec/8970b83323e16aa95034da175454843947376614d6d5e9627f
Successfully built pyswarm


# Importacao dos dados

In [2]:
import pandas as pd
import os
import numpy as np
import random

In [3]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
caminho_dados = '/content/drive/Shareddrives/IC - Otimizacao computacional para inflacao/Dados/dados_USA'

# Listar os arquivos dentro da pasta "Dados"
arquivos_dados = os.listdir(caminho_dados)

print(arquivos_dados)


['Untitled0.ipynb', 'dados_macroeconomicos_usa.csv']


In [5]:
# Caminho para o arquivo CSV
caminho_arquivo = os.path.join(caminho_dados, 'dados_macroeconomicos_usa.csv')

# Ler o arquivo CSV
df = pd.read_csv(caminho_arquivo)

In [6]:
df = df.set_index('DATE')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 180 entries, 2010-01-01 to 2024-12-01
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   180 non-null    int64  
 1   Inflacao_CPI                 180 non-null    float64
 2   Inflacao_CPI_Core            180 non-null    float64
 3   PCE_Core                     180 non-null    float64
 4   Desemprego                   180 non-null    float64
 5   Desemprego_menos_27_semanas  180 non-null    float64
 6   Taxa_Juros_Fed               180 non-null    float64
 7   Emprego_Total_Nao_Agricola   180 non-null    int64  
 8   Permissoes_Construcao        180 non-null    int64  
 9   Salario_Medio                180 non-null    float64
 10  Salario_Medio_Real_Hora      180 non-null    float64
 11  Producao_Industrial          180 non-null    float64
 12  Capacidade_Instalada         180 non-null    float64
 13  PCE      

# Importacao das ferramentas para implementacao do PSO

In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from pyswarm import pso  # Biblioteca para PSO

# Funcao fitness

In [8]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# 📌 Função de fitness (não foi alterada)
def feature_selection_fitness(mask, X_train_scaled, y_train, X_test_scaled, y_test, model, metric_func):
    """
    Função de avaliação para PSO com modelo passado como argumento e métrica.
    """
    mask = np.round(mask).astype(int)  # Converter para 0 ou 1
    selected_features = X_train_scaled.columns[mask == 1]  # Selecionar as features

    if len(selected_features) == 0:  # Caso não tenha selecionado nenhuma feature
        return 9999  # Penalização por não selecionar nenhuma feature

    # Treinar o modelo com as features selecionadas
    model.fit(X_train_scaled[selected_features], y_train)

    # Fazer previsões
    y_pred = model.predict(X_test_scaled[selected_features])

    # Calcular a métrica
    score = metric_func(y_test, y_pred)

    return score  # Retorna a métrica calculada (MAE, MSE, etc.)

# Divisao dos dados e normalizacao dos dados

In [9]:

# 📌 Preparação dos dados
X = df.drop(columns=['Inflacao_CPI'])  # Removendo Inflacao_CPI das features
y = df['Inflacao_CPI']

# Divisão entre treino e teste (SÉRIE TEMPORAL, sem shuffle)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

Normalizacao dos dados(media 0 e desvio padrao 1)

In [10]:
# 📌 Normalização dos dados
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)


# Implementacao usando lasso na funcao fitness

In [11]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error
from pyswarm import pso
import numpy as np
import pandas as pd

# 📌 Garantir que X_train_scaled e X_test_scaled são DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# 📌 Função de fitness com Lasso e restrição de até 5 features
def lasso_feature_selection_fitness(mask, X_train, y_train, X_test, y_test, model, metric):
    mask = np.round(mask).astype(int)

    num_selected = mask.sum()
    if num_selected > 5 or num_selected == 0:
        return 1e6  # Penalização alta se regra for violada

    X_train_selected = X_train.iloc[:, mask == 1]
    X_test_selected = X_test.iloc[:, mask == 1]

    model.fit(X_train_selected, y_train)
    y_pred = model.predict(X_test_selected)

    return metric(y_test, y_pred)  # Lembre-se: MAE menor = melhor

# 📌 Parâmetros do PSO
num_features = X_train_scaled.shape[1]
lb = [0] * num_features  # 0 = feature desligada
ub = [1] * num_features  # 1 = feature ligada

# 📌 Modelo de regressão
lasso_model = Lasso(alpha=0.1, random_state=42)

# 📌 Executar PSO
best_mask, _ = pso(
    lambda mask: lasso_feature_selection_fitness(
        mask,
        X_train_scaled, y_train,
        X_test_scaled, y_test,
        lasso_model,
        mean_absolute_error
    ),
    lb, ub,
    swarmsize=10,
    maxiter=20,
    debug=True
)

# 📌 Arredondar e garantir que no máximo 5 features sejam selecionadas
best_mask = np.round(best_mask).astype(int)

if best_mask.sum() > 5:
    indices = np.argsort(-best_mask)[:5]  # Top 5 "mais ativadas"
    new_mask = np.zeros_like(best_mask)
    new_mask[indices] = 1
    best_mask = new_mask

# 📌 Resultado final
selected_features = X_train_scaled.columns[best_mask == 1]
print("🔹 Melhor conjunto de features selecionado pelo PSO (máx. 5):", list(selected_features))


No constraints given.
Best after iteration 1: [0.74972254 0.41185201 0.08563136 0.47376142 0.90923416 0.9681228
 0.6577441  0.71537888 0.50912934 0.09639785 0.3958046  0.77258503
 0.49690196 0.13224313 0.0360546  0.56940984 0.45883287 0.29322501
 0.12997594] 1000000.0
Best after iteration 2: [0.74972254 0.41185201 0.08563136 0.47376142 0.90923416 0.9681228
 0.6577441  0.71537888 0.50912934 0.09639785 0.3958046  0.77258503
 0.49690196 0.13224313 0.0360546  0.56940984 0.45883287 0.29322501
 0.12997594] 1000000.0
Best after iteration 3: [0.74972254 0.41185201 0.08563136 0.47376142 0.90923416 0.9681228
 0.6577441  0.71537888 0.50912934 0.09639785 0.3958046  0.77258503
 0.49690196 0.13224313 0.0360546  0.56940984 0.45883287 0.29322501
 0.12997594] 1000000.0
Best after iteration 4: [0.74972254 0.41185201 0.08563136 0.47376142 0.90923416 0.9681228
 0.6577441  0.71537888 0.50912934 0.09639785 0.3958046  0.77258503
 0.49690196 0.13224313 0.0360546  0.56940984 0.45883287 0.29322501
 0.12997594] 